In [1]:
import os 
os.environ['dir'] = "/Users/halimeh/Desktop/DDB-NER"

os.chdir(os.environ['dir'])

print("Current Working Directory: ", os.getcwd())

Current Working Directory:  /Users/halimeh/Desktop/DDB-NER


# Load Model

In [9]:
import ipywidgets as widgets
import tqdm as notebook_tqdm
from german_ner.GermanNER import GermanNerModel
from functions.utils import *

model_name = "mschiesser/ner-bert-german" 

label_suffixes = ["-PER",  "-ORG"]
access_token = read_access_token_from_file("../hf_access_token.txt")
ner_model  = GermanNerModel(model_name)


## Example NER

In [12]:
#Example 

input_text = "Angela Merkel ist die Bundeskanzlerin von Deutschland."


input_text = "Musical nach Hans Christian Andersen \
Zwischen Traumwelt und Wirklichkeit: Im Musical-Highlight „Schneekönigin“ wird das Publikum Teil eines spannenden Abenteuers mit zauberhaften Wesen und unbegrenzten Möglichkeiten. Eigens komponierte Musicalsongs, jede Menge Humor und ganz viel Herz sorgen für ein unverwechselbares Live-Erlebnis für die ganze Familie! Das Theater Liberi inszeniert das bekannte Märchen von Hans Christian Andersen als moderne Musicaladaption. \
Das professionelle Musical-Ensemble erzählt die Geschichte von der Einsamkeit einer Außenseiterin und ihrem Wunsch nach Bewunderung und Akzeptanz, aber auch von Mut und bedingungsloser Freundschaft. Dieser Kontrast spiegelt sich auch im Bühnenbild wider, das die Grenzen zwischen Fantasie und Realität verschwimmen lässt. Musikalisch wird dem Publikum eine rasante Reise durch verschiedene Genres mit großen Arrangements geboten, abgerundet von eindrucksvollen Choreografien und einem außergewöhnlichen Lichtdesign.!"


input_text = "Marie Antoinette oder Kuchen für Alle."

entities = ner_model.perform_ner(label_suffixes, input_text)
print(entities)

if entities:
    print("Named Entities:")
    for token, label in entities:
        print(f"Token: {token}\tLabel: {label}")
else:
    print("NER processing failed.")


print("\n".join(concatenate_entities(entities)[0]))


concatenate_entities(entities)

[('Marie', 'B-PER'), ('Antoinette', 'I-PER')]
Named Entities:
Token: Marie	Label: B-PER
Token: Antoinette	Label: I-PER
Marie Antoinette


(['Marie Antoinette'], ['PER'])

## 1. Example Organisation From DDB

In [9]:
from ddbcaller._ddbcaller_class import DDBCaller
import os 
import pprint
from functions.utils import *
import pprint


print(os.getcwd())
os.environ['DDB_API_KEY']= read_access_token_from_file(token_file_path= "./ddb_access_token.txt")

ddb = DDBCaller(os.environ['DDB_API_KEY'])

/Users/halimeh/Desktop/DDB-NER


## Example extract gnd infos

In [10]:
#Extract gnd infos

pprint.pprint(extract_gnd_info(ddb.get_organisation("Museum Huelsmann", {"city_de": "Bielefeld"})))

{'numberOfResults': 2,
 'results': [{'city_de': 'Bielefeld',
              'gnd_id': 'http://d-nb.info/gnd/10163060-8',
              'label': 'Museum Huelsmann',
              'state_de': 'Nordrhein-Westfalen',
              'type': 'gnd-organization'},
             {'city_de': 'Bielefeld',
              'gnd_id': 'http://d-nb.info/gnd/2162741-1',
              'label': 'Kunstgewerbesammlung der Stadt Bielefeld -Stiftung '
                       'Huelsmann',
              'state_de': 'Nordrhein-Westfalen',
              'type': 'gnd-organization'}],
 'type': 'gnd-organization'}


## Example return best match

In [9]:
print("sorted_results: ")

pprint.pprint(sorted_jaccard_distance("Museum Huelsmann", extract_gnd_info(ddb.get_organisation("Museum Huelsmann", {"city_de": "Bielefeld"}))))

sorted_results: 
{'Result_0': {'Jaccard-Distance': 0.0,
              'gnd_id': 'http://d-nb.info/gnd/10163060-8',
              'label': 'Museum Huelsmann',
              'type': 'gnd-organization'},
 'Result_1': {'Jaccard-Distance': 0.8571428571428572,
              'gnd_id': 'http://d-nb.info/gnd/2162741-1',
              'label': 'Kunstgewerbesammlung der Stadt Bielefeld -Stiftung '
                       'Huelsmann',
              'type': 'gnd-organization'}}


## Return DDB URL For Exact Match 

In [10]:
#Return ddb url for best match 
results =  sorted_jaccard_distance("Museum Huelsmann", extract_gnd_info(ddb.get_organisation("Museum Huelsmann", {"city_de": "Bielefeld"})))

find_jaccard_best_match("Museum Huelsmann", results)

'https://www.deutsche-digitale-bibliothek.de/organization/gnd/10163060-8'

## 2. Example search for a person

In [11]:
#Get all people matching to "Hans Christian Andersen"

pprint.pprint(ddb.get_person("Hans Christian Andersen", {}))

{'correctedQuery': '',
 'entities': [],
 'facets': [{'facetValues': [],
             'field': 'person_place_fct',
             'numberOfFacets': 0},
            {'facetValues': [],
             'field': 'person_name_fct',
             'numberOfFacets': 0},
            {'facetValues': [],
             'field': 'person_gender_fct',
             'numberOfFacets': 0},
            {'facetValues': [],
             'field': 'person_occupation_fct',
             'numberOfFacets': 0}],
 'fulltexts': [],
 'highlightedTerms': [],
 'nextCursorMark': '',
 'numberOfResults': 3,
 'randomSeed': '',
 'results': [{'docs': [{'count': 523,
                        'dateOfBirth_de': '2. April 1805',
                        'dateOfBirth_en': '2. April 1805',
                        'dateOfDeath_de': '4. August 1875',
                        'dateOfDeath_en': '4. August 1875',
                        'highlighting': {'preferredName': ['<match>Hans</match> '
                                                    

## Example Extract gnd infos

In [12]:
#Extract gnd infos

pprint.pprint(extract_gnd_info(ddb.get_person("Hans Christian Andersen", {})))

{'numberOfResults': 3,
 'results': [{'dateOfBirth_de': '2. April 1805',
              'gnd_id': 'http://d-nb.info/gnd/118502794',
              'label': 'Hans Christian Andersen',
              'placeOfBirth': ['Odense'],
              'professionOrOccupation': ['Schriftsteller',
                                         'Märchenerzähler',
                                         'Librettist'],
              'type': 'person'},
             {'dateOfBirth_de': '',
              'gnd_id': 'http://d-nb.info/gnd/137009852',
              'label': 'Hans Christian Andersen',
              'placeOfBirth': '',
              'professionOrOccupation': '',
              'type': 'person'},
             {'dateOfBirth_de': '',
              'gnd_id': 'http://d-nb.info/gnd/1063008514',
              'label': 'Hans Christian H. Andersen',
              'placeOfBirth': '',
              'professionOrOccupation': ['Archäologe'],
              'type': 'person'}],
 'type': 'person'}


## Return DDB URL For Exact Match 

In [13]:
#Return ddb url for best match 
query = "Hans Christian Andersen"
ddb_url = find_jaccard_best_match(query, sorted_jaccard_distance(query, extract_gnd_info(ddb.get_person(query, {}))))
ddb_url

'https://www.deutsche-digitale-bibliothek.de/person/gnd/118502794'

### Sorted Jaccard Results

In [14]:
sorted_jaccard_distance(query, extract_gnd_info(ddb.get_person(query, {})))

{'Result_0': {'label': 'Hans Christian Andersen',
  'gnd_id': 'http://d-nb.info/gnd/118502794',
  'Jaccard-Distance': 0.0,
  'type': 'person'},
 'Result_1': {'label': 'Hans Christian Andersen',
  'gnd_id': 'http://d-nb.info/gnd/137009852',
  'Jaccard-Distance': 0.0,
  'type': 'person'},
 'Result_2': {'label': 'Hans Christian H. Andersen',
  'gnd_id': 'http://d-nb.info/gnd/1063008514',
  'Jaccard-Distance': 0.25,
  'type': 'person'}}

## Best Match Based On Cosine Similarity Score

In [15]:
query = "Hans Christian Andersen" 

ddb_url ,_ = find_cosine_best_match(ner_model, query, extract_gnd_info(ddb.get_person(query, {})))
ddb_url

'https://www.deutsche-digitale-bibliothek.de/person/gnd/118502794'

## Example Verfiy match against Deutsche Nationalbibliothek

In [16]:
# Verfiy match against Deutsche Nationalbibliothek

from functions.callLobidGndApi import *
from functions.utils import *


query = "Hans Christian Andersen"
ddb_url = find_jaccard_best_match(query, sorted_jaccard_distance(query, extract_gnd_info(ddb.get_person(query, {}))))
query = ddb_url.split("/")[-1]


entity_name = "Hans Christian Andersen"
ddb_url = 'https://www.deutsche-digitale-bibliothek.de/person/gnd/118502794'
ddb_name = 'Deutsche Digitale Bibliothek'
query = ddb_url.split("/")[-1]


match_names(get_lodib_data(query), entity_name=entity_name, ner_model=ner_model)

True

**************************************************************************************************

# Complete Logic

In [5]:
import os 
os.environ['dir'] = "/Users/halimeh/Desktop/DDB-NER"

os.chdir(os.environ['dir'])

print("Current Working Directory: ", os.getcwd())

import time
from german_ner.GermanNER import GermanNerModel
from functions.utils import *
from ddbcaller._ddbcaller_class import DDBCaller
from functions.callLobidGndApi import *
from functions.utils import *

from pprint import pprint

print(os.getcwd())
os.environ['DDB_API_KEY']= read_access_token_from_file(token_file_path= "./ddb_access_token.txt")

ddb = DDBCaller(os.environ['DDB_API_KEY'])


import ipywidgets as widgets
import tqdm as notebook_tqdm
from german_ner.GermanNER import GermanNerModel
from functions.utils import *

model_name = "mschiesser/ner-bert-german" 

label_suffixes = ["-PER",  "-ORG"]
access_token = read_access_token_from_file("../hf_access_token.txt")
ner_model  = GermanNerModel(model_name)


Current Working Directory:  /Users/halimeh/Desktop/DDB-NER
/Users/halimeh/Desktop/DDB-NER


In [6]:
def normalize_text(text: str ) -> str:
    import re

    text = re.sub(r"[^\w\s]", "", text)
    return text

def find_cosine_best_match(ner_model, query: str, gnd_info: dict) -> str:
    """
    Return the best match for the given query.

    Args:
        query (str): The query for which a best match is to be determined.
        gnd_info (dict): A dictionary containing sorted gnd info based on jaccard similarity.

    Returns:
        str: The URL of the best match or None if no match is found.

    The function returns the DDB URL of the best match based on the document type in the GND information.
    If no match is found, it returns None.
    """

    if gnd_info is None or query is None:
        return None
    list_of_results = []
    for result in gnd_info['results']:
        for key, value in result.items():
            if key == 'label':
                list_of_results.append(value)

    if list_of_results == []:
        return None

    best_index, max_similarity = ner_model.find_best_cosine_similarity(query, list_of_results)
    return get_ddb_url(gnd_info['results'][best_index]), max_similarity    
        


def match_names(data, entity_name, ner_model) -> bool: 

    min_simalarity = 0.9

    if data['member'] == []:
        return False

    if data['member'][0]['preferredName'] is not None:
        preferredName = data['member'][0]['preferredName']

        print('preferredName',data['member'][0]['preferredName'], preferredName)

        if entity_name in [preferredName]:
            return True
        
        else:
            _, max_similarity = ner_model.find_best_cosine_similarity(entity_name, [preferredName])
            print('max_similarity', max_similarity)
            if max_similarity >=  min_simalarity:
                return max_similarity
            else:
                return False
    
     
    elif data['member'][0]['preferredNameEntityForThePerson'] is not None:
        print('preferredNameEntityForThePerson' ,data['member'][0]['preferredNameEntityForThePerson'])
        if 'forename' and 'surname' in data['member'][0]['preferredNameEntityForThePerson'].keys():
            temp = data['member'][0]['preferredNameEntityForThePerson']['forename'][0] + " " + data['member'][0]['preferredNameEntityForThePerson']['surname'][0] 
            if entity_name in temp:
                return True
            
    elif data['member'][0]['variantName'] is not None:
        variantName = data['member'][0]['variantName']
        if entity_name in variantName:
            return True
        else:
            _, max_similarity = ner_model.find_best_cosine_similarity(entity_name, variantName)
            if max_similarity >=  min_simalarity:
                return max_similarity
            else:
                return False
    else:
        return False
    
def match_against_dnb(ddb_url, entity_name, ner_model):
    print("\n * Validate Result by mathcing entity name with Deutsche Nationalbibliothek Running ...")

    if isinstance(ddb_url, str) == True:
        query = ddb_url.split("/")[-1]
        url = f'https://d-nb.info/gnd/{query}/about'

        match= match_names(get_lodib_data(query), entity_name, ner_model)    # ddb_name, ddb_url,

        if match is not None:
            print(f" * Match for entity name {entity_name} from Deutsche Nationalbibliothek found with score: ", match, "\n", " * DNB URL: ", url)
        else:
            print(' * No match from Deutsche Nationalbibliothek found')
         

In [9]:
user_input = "Marie Antoinette oder Kuchen für Alle."

if user_input:
            
            entities = ner_model.perform_ner(label_suffixes, '"' + user_input + '"')
            already_seen = list()

            if entities:

                types = concatenate_entities(entities)[0]
                values = concatenate_entities(entities)[1]

                for entity,label in zip(types, values):
                    entity = normalize_text(entity)
                    if entity in already_seen:
                        continue 
                    
                    print( '- Entity Type:', label, '| Entity Value:', entity)
                    already_seen.append(entity)

                    if label == "PER":
                        data = ddb.get_person(entity, {})
                    elif label == "ORG":
                        data = ddb.get_organisation(entity, {})
                    else:
                        data = ddb.get_query(entity, {})

                
                    if data is None:
                        print(" * No match from Deutsche Digitale Bibliothek found.")
                    
                    elif data['numberOfResults'] == 0:
                        print(" * No match from Deutsche Digitale Bibliothek found.")
                        
                    else:
                        sorted_results =  sorted_jaccard_distance(entity,  extract_gnd_info(data))
                        ddb_url = find_jaccard_best_match(entity, sorted_results)

                        if ddb_url is not None:
                            print(' * Best match from DDB after Jaccard Distance: ',ddb_url)
                            
                        else:
                            print('* No exact match was found, running semantic similarity search on entities ...')
                            gnd_info = extract_gnd_info(data)
                            similarity_threshold = .90

                            ddb_url, max_similarity = find_cosine_best_match(ner_model, entity, gnd_info)
                            if max_similarity >= similarity_threshold:
                                print('* Best match from DDB after Semantic Similarity: ',ddb_url, "with similarity score: ", max_similarity)
                                
                        
                        match_against_dnb(ddb_url, entity, ner_model)


- Entity Type: PER | Entity Value: Marie Antoinette
* No exact match was found, running semantic similarity search on entities ...
* Best match from DDB after Semantic Similarity:  https://www.deutsche-digitale-bibliothek.de/person/gnd/118577905 with similarity score:  0.96

 * Validate Result by mathcing entity name with Deutsche Nationalbibliothek Running ...
preferredName Marie Antoinette, Frankreich, Königin Marie Antoinette, Frankreich, Königin
max_similarity 0.96
 * Match for entity name Marie Antoinette from Deutsche Nationalbibliothek found with score:  0.96 
  * DNB URL:  https://d-nb.info/gnd/118577905/about
